# nnU-Net-style 3D segmentation for glioma (Kaggle)

Questo notebook implementa una **nnU-Net 3D** (PlainConvUNet auto-configurata: InstanceNorm + LeakyReLU, feature 32→320, deep supervision, topologia derivata dalla patch size) mantenendo **identici** al notebook 3D U-Net MONAI:

| Elemento | Come è mantenuto |
|---|---|
| Manifest / dev-test | stesso CSV, `split in {train, validation}` = 1003 pazienti di sviluppo, 177 test esclusi |
| **Split 5-fold** | `StratifiedKFold(5, shuffle=True, random_state=42)` su `Cohort ID` (fallback `KFold`), stessa costruzione di `cv_df` → stessi fold |
| Label mapping | `{0,1,2,4} → {0,1,2,3}` |
| Normalizzazione | clip percentili 0.5–99.5 sui voxel ≠ 0 + z-score sui voxel ≠ 0, background = 0 |
| Patch | 64³, 2 patch/paziente/epoca, 75% centrate su un voxel tumorale, sampler raggruppato per paziente |
| Augmentation | flip sui 3 assi, rot90 nel piano (0,1), scale/shift per canale (stessi parametri) |
| Loss | 0.5·CE + 0.5·Soft Dice (background escluso dal Dice) |
| Training | AdamW lr 1e-4, wd 1e-5, batch 2, max 50 epoche, early stopping patience 10 su val loss, FP16 |
| Validazione | volume intero (sliding window), 1 volume/paziente, best checkpoint per val loss |
| Metriche | Dice, IoU (classi 1–3, vuoto/vuoto = 1), + HD e HD95 |

**Differenze rispetto alla 3D U-Net (necessarie per essere "nnU-Net")**: architettura nnU-Net, deep supervision (pesi 1, ½, ¼, 0 normalizzati), preprocessing salvato su disco in `.npy` ritagliato sul bounding box non-zero (così la lettura delle patch è veloce).

In [ ]:
# ================================================================
# CELL 1 - IMPORTS
# ================================================================
import os, gc, json, time, random, shutil, subprocess, sys, warnings, contextlib
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # zittisce i log C++ di TensorFlow/absl (non servono a PyTorch)
os.environ["GLOG_minloglevel"] = "3"

@contextlib.contextmanager
def quiet_native_stderr():
    """Silenzia i messaggi stampati dalle librerie native (fd 2) durante gli import.
    Le eccezioni Python restano visibili."""
    devnull = os.open(os.devnull, os.O_WRONLY)
    saved = os.dup(2)
    try:
        os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved); os.close(devnull)

with quiet_native_stderr():
    import numpy as np
    import pandas as pd
    import nibabel as nib
    import matplotlib.pyplot as plt
    from scipy import ndimage
    from sklearn.model_selection import StratifiedKFold, KFold
    from tqdm.auto import tqdm

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader

    try:
        import monai
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "monai"])
        import monai
    # (i messaggi "nvjpeg2k / nvtiff failed to load" sono librerie opzionali di MONAI per JPEG2000/TIFF: irrilevanti per NIfTI)
    from monai.inferers import sliding_window_inference

warnings.filterwarnings("ignore")
print("MONAI", monai.__version__, "| torch", torch.__version__)

In [ ]:
# ================================================================
# CELL 2 - CONFIGURATION
# ================================================================
IMAGES_DIR    = Path("/kaggle/input/datasets/laurapitton/training-images/Training_images")
MANIFEST_PATH = Path("/kaggle/input/datasets/liviapantarotto/segmentation-manifest/segmentation_manifest.csv")

OUTPUT_DIR     = Path("/kaggle/working/glioma_nnunet")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULTS_DIR    = OUTPUT_DIR / "results"
for d in (OUTPUT_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Dati preprocessati (~25-30 GB in float16): /kaggle/working ha solo ~20 GB, /kaggle/temp di più.
PREPROC_DIR = Path("/kaggle/temp/glioma_nnunet_preprocessed")
try:
    PREPROC_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    PREPROC_DIR = OUTPUT_DIR / "preprocessed"
    PREPROC_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- riproducibilità / hardware ----------------
SEED   = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
set_seed(SEED)

# ---------------- cross-validation (identica alla 3D U-Net) ----------------
N_FOLDS      = 5
FOLDS_TO_RUN = [0, 1, 2, 3, 4]     # su Kaggle conviene 1-2 fold per sessione (limite ~9-12 h)

# ---------------- dati / training (identici alla 3D U-Net) ----------------
MODALITIES   = ["FLAIR", "T1", "T1CE", "T2"]
NUM_INPUT_CHANNELS = 4
NUM_CLASSES  = 4
PATCH_SIZE   = (64, 64, 64)
BATCH_SIZE   = 2
NUM_WORKERS  = 4
MAX_EPOCHS   = 50
PATIENCE     = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-5
SAMPLES_PER_PATIENT     = 2
TUMOR_PATCH_PROBABILITY = 0.75
LOWER_PERCENTILE, UPPER_PERCENTILE = 0.5, 99.5
LABEL_MAPPING = {0: 0, 1: 1, 2: 2, 4: 3}

# ---------------- opzioni nnU-Net ----------------
BASE_FEATURES    = 32     # nnU-Net default
MAX_FEATURES     = 320    # nnU-Net default (3D)
MIN_FEATURE_MAP  = 4      # regola nnU-Net per il numero di stage
DEEP_SUPERVISION = True
OPTIMIZER   = "adamw"     # "adamw" = come la 3D U-Net | "sgd" = nnU-Net nativo (SGD nesterov 0.99, lr 1e-2)
USE_POLY_LR = False       # True = schedule poly (1 - ep/max)^0.9 tipico di nnU-Net

# ---------------- validazione ----------------
VAL_OVERLAP   = 0.25      # nnU-Net usa 0.5 (più lento); alza a 0.5 per la valutazione finale
SW_BATCH_SIZE = 4
VAL_EVERY     = 2      # valida ogni N epoche: UNICA ottimizzazione di tempo (1 = come prima, validazione a ogni epoca)
USE_DATAPARALLEL = True   # come nella versione originale: usa tutte le GPU disponibili
MAX_HOURS_PER_FOLD = None # es. 4.0 per fermare il fold dopo 4 h e tenere il best checkpoint

RUN_TEST_EVAL = False     # cella finale: ensemble dei fold sul test set indipendente

# ---------------- controllo GPU (fallisce subito se il training girerebbe su CPU) ----------------
ALLOW_CPU = False
if DEVICE.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    supported = torch.cuda.get_arch_list()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    if arch not in supported and f"compute_{cap[0]}{cap[1]}" not in supported:
        raise RuntimeError(
            f"Questa build di PyTorch ({torch.__version__}) non supporta la GPU {arch} (build per: {supported}). "
            "In Kaggle scegli 'GPU T4 x2' (non P100) in Settings > Accelerator, oppure installa una build di torch con cu126/cu128.")
    _ = (torch.zeros(1, device=DEVICE) + 1).item()      # test di un kernel reale
elif not ALLOW_CPU:
    raise RuntimeError("GPU non disponibile: in Kaggle apri Settings > Accelerator e scegli GPU T4 x2, poi riavvia la sessione. "
                       "(Imposta ALLOW_CPU = True solo per test di debug.)")

print("Device:", DEVICE, "| AMP:", USE_AMP)
print("Preprocessed data dir:", PREPROC_DIR)

In [ ]:
# ================================================================
# CELL 3 - MANIFEST, DEVELOPMENT SET, TEST SET
#   Cerca automaticamente cartella MRI e manifest sotto /kaggle/input se i percorsi
#   della Cell 2 non esistono, e riallinea i percorsi scritti nel CSV.
# ================================================================
INPUT_ROOT = Path("/kaggle/input")

def find_in_input(name, want_dir):
    for root, dirs, files in os.walk(INPUT_ROOT):
        if want_dir and name in dirs:
            return Path(root) / name
        if not want_dir and name in files:
            return Path(root) / name
    return None

def show_input_tree(max_depth=3):
    print(f"\nContenuto di {INPUT_ROOT}:")
    base = len(INPUT_ROOT.parts)
    for root, dirs, _ in os.walk(INPUT_ROOT):
        depth = len(Path(root).parts) - base
        if depth >= max_depth:
            dirs[:] = []
            continue
        print("  " * depth + (Path(root).name or str(INPUT_ROOT)) + "/")

if not MANIFEST_PATH.exists():
    found = find_in_input(MANIFEST_PATH.name, want_dir=False)
    if found is None:
        show_input_tree()
        raise FileNotFoundError(
            f"Manifest '{MANIFEST_PATH.name}' non trovato in {INPUT_ROOT}. Aggiungi il dataset del manifest "
            "al notebook (pannello destro > Input > Add Input) oppure imposta MANIFEST_PATH nella Cell 2.")
    print("Manifest trovato in:", found)
    MANIFEST_PATH = found

if not IMAGES_DIR.exists():
    found = find_in_input(IMAGES_DIR.name, want_dir=True)
    if found is not None:
        print("Cartella MRI trovata in:", found)
        IMAGES_DIR = found
    else:
        print(f"ATTENZIONE: cartella '{IMAGES_DIR.name}' non trovata. Procedo solo se i percorsi nel manifest sono validi.")

manifest = pd.read_csv(MANIFEST_PATH)
REQUIRED_COLUMNS = ["patient_folder", "FLAIR_path", "T1_path", "T1CE_path", "T2_path",
                    "MASK_path", "split", "Cohort ID"]
missing = [c for c in REQUIRED_COLUMNS if c not in manifest.columns]
assert not missing, f"Missing columns in manifest: {missing}"

# riallinea i percorsi del CSV se puntano a un'altra posizione di Training_images
PATH_COLUMNS = ["FLAIR_path", "T1_path", "T1CE_path", "T2_path", "MASK_path"]
def resolve_path(p):
    p = str(p)
    if Path(p).is_file():
        return p
    marker = IMAGES_DIR.name + "/"
    if marker in p:
        cand = IMAGES_DIR / p.split(marker, 1)[1]
        if cand.is_file():
            return str(cand)
    return p
for c in PATH_COLUMNS:
    manifest[c] = manifest[c].map(resolve_path)

assert len(manifest) == 1180, f"Expected 1180 patients, found {len(manifest)}"
assert manifest["patient_folder"].is_unique

development_df = manifest[manifest["split"].isin(["train", "validation"])].copy().reset_index(drop=True)
test_df        = manifest[manifest["split"] == "test"].copy().reset_index(drop=True)
assert len(development_df) == 1003 and len(test_df) == 177
assert set(development_df["patient_folder"].astype(str)).isdisjoint(set(test_df["patient_folder"].astype(str)))

missing_files = [(r["patient_folder"], c, r[c]) for _, r in manifest.iterrows() for c in PATH_COLUMNS
                 if not (isinstance(r[c], str) and Path(r[c]).is_file())]
if missing_files:
    show_input_tree()
    print("Esempio di percorso mancante:", missing_files[0])
    raise FileNotFoundError(f"{len(missing_files)} file MRI/maschera non trovati (primi 5: {missing_files[:5]}). "
                            "Verifica che il dataset delle immagini sia aggiunto al notebook.")
print("Development:", len(development_df), "| Test:", len(test_df), "| all files found")

In [ ]:
# ================================================================
# CELL 4 - 5-FOLD CROSS-VALIDATION (IDENTICO ALLA 3D U-NET)
# ================================================================
cv_df = development_df.copy().reset_index(drop=True)

stratification_labels = cv_df["Cohort ID"].fillna("Unknown").astype(str)

if stratification_labels.value_counts().min() >= N_FOLDS:
    print("Using StratifiedKFold by Cohort ID.")
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    split_iterator = cv.split(cv_df, stratification_labels)
else:
    print("Warning: a cohort has fewer than N_FOLDS patients. Using KFold.")
    cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    split_iterator = cv.split(cv_df)

cv_df["fold"] = -1
for fold, (_, val_indices) in enumerate(split_iterator):
    cv_df.loc[val_indices, "fold"] = fold

assert (cv_df["fold"] >= 0).all() and cv_df["patient_folder"].is_unique
cv_df.to_csv(OUTPUT_DIR / "5fold_manifest.csv", index=False)

# integrity checks
all_val = []
for fold in range(N_FOLDS):
    tr = set(cv_df[cv_df["fold"] != fold]["patient_folder"].astype(str))
    va = set(cv_df[cv_df["fold"] == fold]["patient_folder"].astype(str))
    assert tr.isdisjoint(va)
    all_val.extend(va)
    print(f"Fold {fold}: Train = {len(tr)}, Validation = {len(va)}")
assert len(all_val) == len(set(all_val)) == len(cv_df)
print("\nCohort distribution per fold:")
display(pd.crosstab(cv_df["fold"], cv_df["Cohort ID"]))

In [ ]:
# ================================================================
# CELL 5 - PREPROCESSING (stessa normalizzazione e label mapping della 3D U-Net)
#   - normalize_mri: percentile clip + z-score sui voxel != 0, background = 0
#   - ritaglio sul bounding box (voxel non-zero in almeno una modalità, o maschera > 0)
#     => fuori dal crop GT e predizione sono background: Dice/IoU/HD identici al volume intero
#   - salvataggio .npy (immagine float16, maschera uint8) per lettura veloce via mmap
# ================================================================
def normalize_mri(volume, lower_pct=LOWER_PERCENTILE, upper_pct=UPPER_PERCENTILE):
    volume = np.asarray(volume, dtype=np.float32)
    nonzero_mask = volume != 0
    if not nonzero_mask.any():
        return np.zeros_like(volume, dtype=np.float32)
    values = volume[nonzero_mask]
    lower = np.percentile(values, lower_pct)
    upper = np.percentile(values, upper_pct)
    clipped = np.clip(volume, lower, upper)
    clipped_values = clipped[nonzero_mask]
    mean, std = clipped_values.mean(), clipped_values.std()
    if std < 1e-8:
        normalized = np.zeros_like(volume, dtype=np.float32)
    else:
        normalized = (clipped - mean) / std
    normalized[~nonzero_mask] = 0
    return normalized.astype(np.float32)


def _atomic_save_npy(path, array):
    tmp = str(path) + ".tmp"
    with open(tmp, "wb") as f:
        np.save(f, array)
    os.replace(tmp, path)


def preprocess_patient(job):
    pid, image_paths, mask_path, out_dir = job
    out_dir = Path(out_dir)
    meta_path = out_dir / f"{pid}.json"
    if meta_path.exists():                       # già fatto (resume)
        return json.load(open(meta_path))

    channels, spacing = [], None
    for p in image_paths:
        nii = nib.load(str(p))
        if spacing is None:
            spacing = [float(z) for z in nii.header.get_zooms()[:3]]
        channels.append(normalize_mri(nii.get_fdata(dtype=np.float32)))
    image = np.stack(channels, axis=0)                      # (4, X, Y, Z)

    mask = nib.load(str(mask_path)).get_fdata(dtype=np.float32)
    if image.shape[1:] != mask.shape:
        raise ValueError(f"Image/mask shape mismatch for {pid}: {image.shape[1:]} vs {mask.shape}")
    unexpected = set(np.unique(mask).astype(int)) - set(LABEL_MAPPING)
    if unexpected:
        raise ValueError(f"Unexpected labels for {pid}: {unexpected}")
    mapped = np.zeros(mask.shape, dtype=np.uint8)
    for o, t in LABEL_MAPPING.items():
        mapped[mask == o] = t

    nz = (image != 0).any(axis=0) | (mapped > 0)
    if nz.any():
        coords = np.where(nz)
        bbox = [[int(c.min()), int(c.max()) + 1] for c in coords]
    else:
        bbox = [[0, s] for s in mask.shape]
    sl = tuple(slice(a, b) for a, b in bbox)
    image_c = np.ascontiguousarray(image[(slice(None),) + sl]).astype(np.float16)
    mask_c  = np.ascontiguousarray(mapped[sl])

    _atomic_save_npy(out_dir / f"{pid}_img.npy", image_c)
    _atomic_save_npy(out_dir / f"{pid}_seg.npy", mask_c)
    meta = {"patient": pid, "orig_shape": list(mask.shape), "bbox": bbox,
            "crop_shape": list(mask_c.shape), "spacing": spacing}
    json.dump(meta, open(meta_path, "w"))      # scritto per ultimo = marker di completamento
    return meta


def preprocess_dataframe(df, workers=4):
    jobs = [(str(r["patient_folder"]), [r[f"{m}_path"] for m in MODALITIES], r["MASK_path"], str(PREPROC_DIR))
            for _, r in df.iterrows()]
    with ProcessPoolExecutor(max_workers=workers) as ex:
        metas = list(tqdm(ex.map(preprocess_patient, jobs), total=len(jobs), desc="Preprocessing"))
    return {m["patient"]: m for m in metas}


free_gb = shutil.disk_usage(PREPROC_DIR).free / 1e9
print(f"Free disk space in {PREPROC_DIR}: {free_gb:.1f} GB (servono ~25-30 GB per 1003 pazienti)")

META = preprocess_dataframe(development_df, workers=min(4, os.cpu_count() or 1))
crop = np.array([m["crop_shape"] for m in META.values()])
print("Original shape (first patient):", next(iter(META.values()))["orig_shape"])
print("Cropped shape min/max per axis:", crop.min(0), crop.max(0))
print("Spacing (first patient):", next(iter(META.values()))["spacing"])

In [ ]:
# ================================================================
# CELL 6 - DATASET, PATCH EXTRACTION, AUGMENTATION, SAMPLER
#   (stessa logica della 3D U-Net, leggendo da .npy con mmap)
# ================================================================
def extract_3d_patch(image, mask, patch_size, tumor_probability=0.75):
    """image: (C,X,Y,Z) mmap/array, mask: (X,Y,Z) array."""
    spatial_shape = mask.shape
    tumor_coords = np.argwhere(mask > 0)
    if len(tumor_coords) > 0 and random.random() < tumor_probability:
        center = tumor_coords[random.randrange(len(tumor_coords))]
    else:
        center = np.array([random.randrange(s) for s in spatial_shape])

    slices = []
    for c, p, dim in zip(center, patch_size, spatial_shape):
        start = int(c - p // 2)
        start = max(0, min(start, dim - p))
        slices.append(slice(start, start + min(p, dim)))
    slices = tuple(slices)

    image_patch = np.asarray(image[(slice(None),) + slices], dtype=np.float32)
    mask_patch  = np.array(mask[slices])

    pad = [((p - s) // 2, (p - s) - (p - s) // 2) if s < p else (0, 0)
           for s, p in zip(mask_patch.shape, patch_size)]
    if any(a or b for a, b in pad):                       # volume più piccolo della patch
        image_patch = np.pad(image_patch, [(0, 0)] + pad, mode="constant")
        mask_patch  = np.pad(mask_patch, pad, mode="constant")
    return image_patch, mask_patch


def augment_patch(image, mask):
    for axis in range(3):                                  # flip
        if random.random() < 0.5:
            image = np.flip(image, axis=axis + 1)
            mask  = np.flip(mask, axis=axis)
    k = random.randint(0, 3)                               # rot90 nel piano (0,1)
    image = np.rot90(image, k=k, axes=(1, 2))
    mask  = np.rot90(mask,  k=k, axes=(0, 1))
    for ch in range(image.shape[0]):                       # scale/shift per modalità
        if random.random() < 0.5:
            image[ch] = image[ch] * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)
    return np.ascontiguousarray(image, dtype=np.float32), np.ascontiguousarray(mask, dtype=np.int64)


class GliomaPatchDataset(Dataset):
    """Training: samples_per_patient patch casuali per paziente (consecutive)."""
    def __init__(self, dataframe, samples_per_patient=2):
        self.pids = [str(p) for p in dataframe["patient_folder"]]
        self.samples_per_patient = samples_per_patient
        assert PATCH_SIZE[0] == PATCH_SIZE[1], "rot90 nel piano (0,1) richiede patch quadrata in X,Y"

    def __len__(self):
        return len(self.pids) * self.samples_per_patient

    def __getitem__(self, index):
        pid = self.pids[index // self.samples_per_patient]
        img = np.load(PREPROC_DIR / f"{pid}_img.npy", mmap_mode="r")
        seg = np.load(PREPROC_DIR / f"{pid}_seg.npy")
        image, mask = extract_3d_patch(img, seg, PATCH_SIZE, TUMOR_PATCH_PROBABILITY)
        image, mask = augment_patch(image, mask)
        return {"image": torch.from_numpy(image), "mask": torch.from_numpy(mask), "patient_id": pid}


class PatientGroupedSampler(torch.utils.data.Sampler):
    """Shuffle a livello di paziente; i sample dello stesso paziente restano consecutivi."""
    def __init__(self, dataset, shuffle=True, seed=42):
        self.dataset, self.shuffle, self.seed, self.epoch = dataset, shuffle, seed, 0
    def set_epoch(self, epoch):
        self.epoch = epoch
    def __iter__(self):
        patients = np.arange(len(self.dataset.pids))
        if self.shuffle:
            np.random.default_rng(self.seed + self.epoch).shuffle(patients)
        spp = self.dataset.samples_per_patient
        return iter([int(p) * spp + j for p in patients for j in range(spp)])
    def __len__(self):
        return len(self.dataset)

# smoke test
_ds = GliomaPatchDataset(cv_df.iloc[:4], SAMPLES_PER_PATIENT)
_s = _ds[0]
print("Patch:", tuple(_s["image"].shape), tuple(_s["mask"].shape), "| labels:", torch.unique(_s["mask"]).tolist())
del _ds, _s

In [ ]:
# ================================================================
# CELL 7 - nnU-Net 3D ARCHITECTURE (PlainConvUNet, auto-configurata)
#   - numero di stage dalla patch size (feature map minima = 4) -> 64^3: 5 stage, 128^3: 6 stage
#   - feature: 32, 64, 128, 256, 320 (cap a 320)
#   - blocchi: Conv3x3x3 -> InstanceNorm -> LeakyReLU(0.01), 2 per stage
#   - downsampling con stride-2 nella prima conv di ogni stage; upsampling con ConvTranspose 2x2x2
#   - deep supervision: una head 1x1x1 per ogni livello del decoder
# ================================================================
class ConvBlock(nn.Sequential):
    def __init__(self, in_c, out_c, stride):
        super().__init__(
            nn.Conv3d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=True),
            nn.InstanceNorm3d(out_c, eps=1e-5, affine=True),
            nn.LeakyReLU(negative_slope=0.01, inplace=True),
        )

class StackedConvs(nn.Sequential):
    def __init__(self, in_c, out_c, stride):
        super().__init__(ConvBlock(in_c, out_c, stride), ConvBlock(out_c, out_c, 1))


class NNUNet3D(nn.Module):
    def __init__(self, in_channels=4, num_classes=4, patch_size=(64, 64, 64),
                 base_features=32, max_features=320, min_feature_map=4, deep_supervision=True):
        super().__init__()
        size, n_stages = min(patch_size), 1
        while size // 2 >= min_feature_map:
            size //= 2
            n_stages += 1
        self.n_stages = n_stages
        self.deep_supervision = deep_supervision
        feats = [min(base_features * 2 ** i, max_features) for i in range(n_stages)]
        self.features = feats

        self.encoder = nn.ModuleList([
            StackedConvs(in_channels if i == 0 else feats[i - 1], feats[i], stride=1 if i == 0 else 2)
            for i in range(n_stages)
        ])
        self.up, self.decoder, self.heads = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for i in range(1, n_stages):                      # i = livello del decoder, dal basso
            low, high = feats[-i], feats[-(i + 1)]
            self.up.append(nn.ConvTranspose3d(low, high, kernel_size=2, stride=2, bias=False))
            self.decoder.append(StackedConvs(2 * high, high, stride=1))
            self.heads.append(nn.Conv3d(high, num_classes, kernel_size=1))

        for m in self.modules():                           # He init (come nnU-Net)
            if isinstance(m, (nn.Conv3d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, a=1e-2)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        skips = []
        for i, stage in enumerate(self.encoder):
            x = stage(x)
            if i < self.n_stages - 1:
                skips.append(x)
        outs = []
        for i, (up, dec, head) in enumerate(zip(self.up, self.decoder, self.heads)):
            x = up(x)
            x = torch.cat([x, skips[-(i + 1)]], dim=1)
            x = dec(x)
            outs.append(head(x))
        outs = outs[::-1]                                  # [full-res, 1/2, 1/4, ...]
        if self.training and self.deep_supervision:
            return tuple(outs)
        return outs[0]


def create_model():
    model = NNUNet3D(NUM_INPUT_CHANNELS, NUM_CLASSES, PATCH_SIZE, BASE_FEATURES,
                     MAX_FEATURES, MIN_FEATURE_MAP, DEEP_SUPERVISION)
    if USE_DATAPARALLEL and torch.cuda.is_available() and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model.to(DEVICE)

def get_state_dict(model):
    return model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

# shape test
_m = create_model()
_inner = _m.module if isinstance(_m, nn.DataParallel) else _m
print("Stages:", _inner.n_stages, "| features:", _inner.features)
print("Parameters:", f"{sum(p.numel() for p in _m.parameters()):,}")
_m.train()
with torch.no_grad():
    _o = _m(torch.randn(2, NUM_INPUT_CHANNELS, *PATCH_SIZE, device=DEVICE))
print("Deep supervision outputs:", [tuple(o.shape[2:]) for o in _o])
del _m, _inner, _o
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ================================================================
# CELL 8 - LOSS (identica alla 3D U-Net) + DEEP SUPERVISION
# ================================================================
class SoftDiceLoss(nn.Module):
    """Soft Dice multi-classe, background (classe 0) escluso."""
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs = torch.softmax(logits, dim=1)
        onehot = F.one_hot(targets.long(), num_classes=num_classes).permute(0, 4, 1, 2, 3).float()
        probs, onehot = probs[:, 1:], onehot[:, 1:]
        probs  = probs.reshape(probs.shape[0], probs.shape[1], -1)
        onehot = onehot.reshape(onehot.shape[0], onehot.shape[1], -1)
        inter = (probs * onehot).sum(dim=2)
        denom = probs.sum(dim=2) + onehot.sum(dim=2)
        dice = (2.0 * inter + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()


class CombinedLoss(nn.Module):
    """0.5 * CrossEntropy + 0.5 * SoftDice."""
    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce_weight, self.dice_weight = ce_weight, dice_weight
        self.ce, self.dice = nn.CrossEntropyLoss(), SoftDiceLoss()
    def forward(self, logits, targets):
        ce, dice = self.ce(logits, targets), self.dice(logits, targets)
        return self.ce_weight * ce + self.dice_weight * dice, ce, dice


class DeepSupervisionLoss(nn.Module):
    """Pesi nnU-Net: 1/2^i, ultimo (risoluzione minore) = 0, poi normalizzati."""
    def __init__(self, base_loss):
        super().__init__()
        self.base_loss = base_loss
    def forward(self, outputs, targets):
        if not isinstance(outputs, (list, tuple)):
            return self.base_loss(outputs, targets)
        w = np.array([1.0 / 2 ** i for i in range(len(outputs))]); w[-1] = 0.0; w /= w.sum()
        total = ce_t = dice_t = 0.0
        for wi, out in zip(w, outputs):
            if wi == 0:
                continue
            t = targets if out.shape[2:] == targets.shape[1:] else \
                F.interpolate(targets[:, None].float(), size=out.shape[2:], mode="nearest")[:, 0].long()
            l, ce, dc = self.base_loss(out, t)
            total, ce_t, dice_t = total + wi * l, ce_t + wi * ce, dice_t + wi * dc
        return total, ce_t, dice_t

print("Losses defined.")

In [ ]:
# ================================================================
# CELL 9 - METRICHE (Dice, IoU come nella 3D U-Net; HD / HD95 aggiunte) + VALIDAZIONE SU VOLUME INTERO
# ================================================================
def dice_iou_per_class(prediction, target, num_classes=NUM_CLASSES):
    """Classi 1..C-1; se predizione e target sono entrambi vuoti -> 1.0 (come nel notebook originale)."""
    dices, ious = [], []
    for c in range(1, num_classes):
        p, t = (prediction == c), (target == c)
        inter = (p & t).sum().float()
        p_sum, t_sum = p.sum().float(), t.sum().float()
        union = (p | t).sum().float()
        dices.append(1.0 if (p_sum + t_sum) == 0 else (2 * inter / (p_sum + t_sum)).item())
        ious.append(1.0 if union == 0 else (inter / union).item())
    return dices, ious


_STRUCT = ndimage.generate_binary_structure(3, 1)

def hausdorff_metrics(pred_c, gt_c, spacing):
    """HD e HD95 (mm) simmetrici, convenzione MONAI (max dei due percentili direzionali).
    entrambi vuoti -> 0.0 ; uno solo vuoto -> NaN (escluso dalle medie)."""
    pa, ga = pred_c.any(), gt_c.any()
    if not pa and not ga:
        return 0.0, 0.0
    if not pa or not ga:
        return np.nan, np.nan
    p_surf = pred_c & ~ndimage.binary_erosion(pred_c, _STRUCT)
    g_surf = gt_c   & ~ndimage.binary_erosion(gt_c,   _STRUCT)
    d_to_g = ndimage.distance_transform_edt(~g_surf, sampling=spacing)[p_surf]
    d_to_p = ndimage.distance_transform_edt(~p_surf, sampling=spacing)[g_surf]
    hd   = float(max(d_to_g.max(), d_to_p.max()))
    hd95 = float(max(np.percentile(d_to_g, 95), np.percentile(d_to_p, 95)))
    return hd, hd95


def load_patient_tensors(pid):
    img = torch.from_numpy(np.load(PREPROC_DIR / f"{pid}_img.npy").astype(np.float32))[None]
    seg = torch.from_numpy(np.load(PREPROC_DIR / f"{pid}_seg.npy").astype(np.int64))[None]
    return img.to(DEVICE), seg.to(DEVICE)


@torch.no_grad()
def sliding_logits(model, image, overlap=None):
    with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
        out = sliding_window_inference(image, PATCH_SIZE, SW_BATCH_SIZE, model,
                                       overlap=overlap if overlap is not None else VAL_OVERLAP,
                                       mode="gaussian")
    return out.float()


@torch.no_grad()
def evaluate(model, df, criterion, surface=False, desc="Validation"):
    """Un volume per paziente, sliding window sul volume intero. Ritorna una riga per paziente."""
    model.eval()
    rows = []
    for pid in tqdm([str(p) for p in df["patient_folder"]], desc=desc, leave=False):
        image, seg = load_patient_tensors(pid)
        logits = sliding_logits(model, image)
        loss, ce, dl = criterion(logits, seg)
        pred = logits.argmax(1)
        dices, ious = dice_iou_per_class(pred, seg)
        row = {"patient": pid, "loss": loss.item(), "ce_loss": ce.item(), "dice_loss": dl.item(),
               "dice": float(np.mean(dices)), "iou": float(np.mean(ious))}
        for c in range(1, NUM_CLASSES):
            row[f"dice_c{c}"], row[f"iou_c{c}"] = dices[c - 1], ious[c - 1]
        if surface:
            p_np, g_np = pred[0].cpu().numpy(), seg[0].cpu().numpy()
            spacing = META[pid]["spacing"] if pid in META else (1.0, 1.0, 1.0)
            for c in range(1, NUM_CLASSES):
                row[f"hd_c{c}"], row[f"hd95_c{c}"] = hausdorff_metrics(p_np == c, g_np == c, spacing)
            row["hd"]   = float(np.nanmean([row[f"hd_c{c}"]   for c in range(1, NUM_CLASSES)])) \
                          if not all(np.isnan(row[f"hd_c{c}"]) for c in range(1, NUM_CLASSES)) else np.nan
            row["hd95"] = float(np.nanmean([row[f"hd95_c{c}"] for c in range(1, NUM_CLASSES)])) \
                          if not all(np.isnan(row[f"hd95_c{c}"]) for c in range(1, NUM_CLASSES)) else np.nan
        rows.append(row)
        del image, seg, logits, pred
    return pd.DataFrame(rows)

print("Metrics defined: Dice, IoU, HD, HD95.")

In [ ]:
# ================================================================
# CELL 10 - FUNZIONI DI TRAINING MEMORY-SAFE E TRAINING SINGOLO FOLD
# ================================================================
import zipfile


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    run = {"loss": 0.0, "ce": 0.0, "dice": 0.0}
    n = 0
    for batch in tqdm(loader, desc="Training", leave=False):
        images = batch["image"]
        masks  = batch["mask"]
        micro_steps = images.shape[0]          # gradient accumulation: identico a batch_size=2 con InstanceNorm

        optimizer.zero_grad(set_to_none=True)
        batch_loss = batch_ce = batch_dice = 0.0

        for i in range(micro_steps):
            img_mb  = images[i : i + 1].to(DEVICE, non_blocking=True)
            mask_mb = masks[i : i + 1].to(DEVICE, non_blocking=True)

            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                outputs = model(img_mb)
                loss, ce, dl = criterion(outputs, mask_mb)
                scaled_loss = loss / micro_steps

            scaler.scale(scaled_loss).backward()

            batch_loss += loss.detach() / micro_steps
            batch_ce   += ce.detach() / micro_steps
            batch_dice += dl.detach() / micro_steps

            del img_mb, mask_mb, outputs, loss, ce, dl, scaled_loss

        scaler.step(optimizer)
        scaler.update()

        run["loss"] += batch_loss
        run["ce"]   += batch_ce
        run["dice"] += batch_dice
        n += 1
        del images, masks

    return {k: (v / n).item() for k, v in run.items()}


@torch.no_grad()
def evaluate_memory_safe(model, df, surface=False, desc="Validation"):
    """Stesso calcolo di evaluate() della Cell 9, ma con sw_batch_size=1 e Dice senza one_hot 5D."""
    model.eval()
    rows = []
    for pid in tqdm([str(p) for p in df["patient_folder"]], desc=desc, leave=False):
        image, seg = load_patient_tensors(pid)
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            logits = sliding_window_inference(
                image, PATCH_SIZE, sw_batch_size=1, predictor=model,
                overlap=VAL_OVERLAP, mode="gaussian"
            )
        logits = logits.float()

        ce = F.cross_entropy(logits, seg)
        probs = torch.softmax(logits, dim=1)
        dice_terms = []
        for c in range(1, NUM_CLASSES):
            p_c = probs[:, c]
            t_c = (seg == c).float()
            inter = (p_c * t_c).sum()
            denom = p_c.sum() + t_c.sum()
            dice_terms.append((2.0 * inter + 1e-5) / (denom + 1e-5))
        dl = 1.0 - torch.stack(dice_terms).mean()
        loss = 0.5 * ce + 0.5 * dl

        pred = logits.argmax(1)
        dices, ious = dice_iou_per_class(pred, seg)
        row = {
            "patient": pid, "loss": loss.item(), "ce_loss": ce.item(), "dice_loss": dl.item(),
            "dice": float(np.mean(dices)), "iou": float(np.mean(ious)),
        }
        for c in range(1, NUM_CLASSES):
            row[f"dice_c{c}"], row[f"iou_c{c}"] = dices[c - 1], ious[c - 1]

        if surface:
            p_np, g_np = pred[0].cpu().numpy(), seg[0].cpu().numpy()
            spacing = META[pid]["spacing"] if pid in META else (1.0, 1.0, 1.0)
            for c in range(1, NUM_CLASSES):
                row[f"hd_c{c}"], row[f"hd95_c{c}"] = hausdorff_metrics(p_np == c, g_np == c, spacing)
            row["hd"] = float(np.nanmean([row[f"hd_c{c}"] for c in range(1, NUM_CLASSES)])) \
                        if not all(np.isnan(row[f"hd_c{c}"]) for c in range(1, NUM_CLASSES)) else np.nan
            row["hd95"] = float(np.nanmean([row[f"hd95_c{c}"] for c in range(1, NUM_CLASSES)])) \
                          if not all(np.isnan(row[f"hd95_c{c}"]) for c in range(1, NUM_CLASSES)) else np.nan

        rows.append(row)
        del image, seg, logits, probs, pred, dice_terms, ce, dl, loss
    return pd.DataFrame(rows)


def create_optimizer(model):
    if OPTIMIZER == "sgd":
        return torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.99, nesterov=True, weight_decay=3e-5)
    return torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def run_single_fold(fold):
    """Allena un singolo fold, salva checkpoint/metriche e crea uno zip pronto da scaricare."""
    summary_path    = RESULTS_DIR / f"fold_{fold}_summary.json"
    history_path    = RESULTS_DIR / f"fold_{fold}_history.csv"
    metrics_path    = RESULTS_DIR / f"fold_{fold}_patient_metrics.csv"
    checkpoint_path = CHECKPOINT_DIR / f"nnunet_fold{fold}_best.pth"

    if summary_path.exists():
        print(f"Fold {fold} già completato -> salto.")
        return json.load(open(summary_path))

    print("\n" + "=" * 70)
    print(f"STARTING FOLD {fold}")
    print("=" * 70)
    set_seed(SEED + fold)

    train_fold_df = cv_df[cv_df["fold"] != fold].copy().reset_index(drop=True)
    val_fold_df   = cv_df[cv_df["fold"] == fold].copy().reset_index(drop=True)
    print(f"Training patients   : {len(train_fold_df)}")
    print(f"Validation patients : {len(val_fold_df)}")

    train_dataset = GliomaPatchDataset(train_fold_df, SAMPLES_PER_PATIENT)
    train_sampler = PatientGroupedSampler(train_dataset, shuffle=True, seed=SEED + fold)
    train_loader  = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        shuffle=False,
        num_workers=min(2, NUM_WORKERS),
        pin_memory=False,
        persistent_workers=False,
    )
    print(f"Training patches per epoch : {len(train_dataset)}")
    print(f"Validation volumes         : {len(val_fold_df)} (volume intero, sliding window)")

    model = create_model()
    if isinstance(model, nn.DataParallel):
        model = model.module
    optimizer = create_optimizer(model)
    base_lr   = optimizer.param_groups[0]["lr"]
    train_criterion = DeepSupervisionLoss(CombinedLoss(ce_weight=0.5, dice_weight=0.5))
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    history, best_val_loss, best_epoch, epochs_without_improvement = [], float("inf"), 0, 0
    fold_start_time = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start_time = time.time()
        print("\n" + "-" * 70)
        print(f"Fold {fold} | Epoch {epoch}/{MAX_EPOCHS}")
        print("-" * 70)

        if USE_POLY_LR:
            for g in optimizer.param_groups:
                g["lr"] = base_lr * (1 - (epoch - 1) / MAX_EPOCHS) ** 0.9

        train_sampler.set_epoch(epoch)
        train_metrics = train_one_epoch(model, train_loader, optimizer, train_criterion, scaler)

        do_val = (epoch % VAL_EVERY == 0) or (epoch == MAX_EPOCHS)
        val_cols = ["loss", "ce_loss", "dice_loss", "dice", "iou"]
        if do_val:
            optimizer.zero_grad(set_to_none=True)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            val_patients = evaluate_memory_safe(model, val_fold_df, surface=False)
            val_metrics  = val_patients[val_cols].mean()
            del val_patients
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            val_metrics = pd.Series({k: np.nan for k in val_cols})

        epoch_time = time.time() - epoch_start_time
        history.append({
            "fold": fold, "epoch": epoch,
            "train_loss": train_metrics["loss"], "train_ce_loss": train_metrics["ce"],
            "train_dice_loss": train_metrics["dice"],
            "val_loss": val_metrics["loss"], "val_ce_loss": val_metrics["ce_loss"],
            "val_dice_loss": val_metrics["dice_loss"],
            "val_dice": val_metrics["dice"], "val_iou": val_metrics["iou"],
            "lr": optimizer.param_groups[0]["lr"], "epoch_time_seconds": epoch_time,
        })

        print(f"\nTrain loss: {train_metrics['loss']:.4f}")
        if do_val:
            print(f"Val loss:   {val_metrics['loss']:.4f} | Val Dice: {val_metrics['dice']:.4f} | Val IoU: {val_metrics['iou']:.4f}")
        else:
            print("Validation skipped (VAL_EVERY).")
        print(f"Epoch time: {epoch_time / 60:.2f} min")

        if do_val and val_metrics["loss"] < best_val_loss:
            best_val_loss, best_epoch, epochs_without_improvement = val_metrics["loss"], epoch, 0
            torch.save({
                "fold": fold, "epoch": epoch,
                "model_state_dict": get_state_dict(model),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss, "best_val_dice": val_metrics["dice"],
                "config": {
                    "patch_size": PATCH_SIZE, "batch_size": BATCH_SIZE,
                    "base_features": BASE_FEATURES, "max_features": MAX_FEATURES,
                    "min_feature_map": MIN_FEATURE_MAP, "deep_supervision": DEEP_SUPERVISION,
                    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
                    "num_classes": NUM_CLASSES, "num_input_channels": NUM_INPUT_CHANNELS,
                },
            }, checkpoint_path)
            print(f"✓ Best checkpoint saved: {checkpoint_path.name}")
        elif do_val:
            epochs_without_improvement += VAL_EVERY
            print(f"No improvement for {epochs_without_improvement} epoch(s).")

        pd.DataFrame(history).to_csv(history_path, index=False)

        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping after {PATIENCE} epochs without improvement.")
            break
        if MAX_HOURS_PER_FOLD and (time.time() - fold_start_time) / 3600 > MAX_HOURS_PER_FOLD:
            print("\nTime budget reached for this fold.")
            break

    del optimizer, scaler, train_loader, train_dataset, train_sampler, train_criterion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ===================== FINE FOLD: valutazione finale con HD / HD95 =====================
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    del ckpt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    final_df = evaluate_memory_safe(model, val_fold_df, surface=True, desc="Final evaluation")
    final_df.insert(0, "fold", fold)
    final_df.to_csv(metrics_path, index=False)

    fold_result = {
        "fold": fold, "best_epoch": best_epoch, "best_val_loss": float(best_val_loss),
        "training_time_minutes": (time.time() - fold_start_time) / 60.0,
    }
    for col in [c for c in final_df.columns if c.startswith(("dice", "iou", "hd")) and c != "dice_loss"]:
        fold_result[f"val_{col}"] = float(np.nanmean(final_df[col]))
    json.dump(fold_result, open(summary_path, "w"), indent=2)

    # ===================== PACCHETTO ZIP PER IL DOWNLOAD RAPIDO =====================
    zip_path = Path("/kaggle/working") / f"fold_{fold}_output.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fpath in [checkpoint_path, history_path, metrics_path, summary_path, OUTPUT_DIR / "5fold_manifest.csv"]:
            if fpath.exists():
                zf.write(fpath, arcname=fpath.relative_to(OUTPUT_DIR))

    print("\n" + "=" * 70)
    print(f"FOLD {fold} COMPLETED")
    print("=" * 70)
    print(f"Best epoch    : {best_epoch}")
    print(f"Best val loss : {best_val_loss:.4f}")
    for k in ("val_dice", "val_iou", "val_hd", "val_hd95"):
        print(f"{k:<14}: {fold_result[k]:.4f}")
    print(f"Training time : {fold_result['training_time_minutes']:.2f} min")
    print(f"\n📦 File pronto da scaricare in Output: {zip_path}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return fold_result


print("Memory-safe training functions and run_single_fold() defined.")

In [ ]:
# ================================================================
# CELL 11A - TRAINING FOLD 0
# ================================================================
fold_0_result = run_single_fold(0)

In [ ]:
# ================================================================
# CELL 11B - TRAINING FOLD 1
# ================================================================
fold_1_result = run_single_fold(1)

In [ ]:
# ================================================================
# CELL 11C - TRAINING FOLD 2
# ================================================================
fold_2_result = run_single_fold(2)

In [ ]:
# ================================================================
# CELL 11D - TRAINING FOLD 3
# ================================================================
fold_3_result = run_single_fold(3)

In [ ]:
# ================================================================
# CELL 11E - TRAINING FOLD 4
# ================================================================
fold_4_result = run_single_fold(4)

In [ ]:
# ================================================================
# CELL 12 - RIEPILOGO CROSS-VALIDATION + CURVE
# ================================================================
summaries = [json.load(open(p)) for p in sorted(RESULTS_DIR.glob("fold_*_summary.json"))]
fold_df = pd.DataFrame(summaries).sort_values("fold")
fold_df.to_csv(RESULTS_DIR / "5fold_summary.csv", index=False)
display(fold_df)

metric_cols = [c for c in fold_df.columns if c.startswith("val_") and c != "val_loss"]
agg = fold_df[metric_cols].agg(["mean", "std"]).T
print(f"\nFold completati: {sorted(fold_df['fold'].tolist())}")
display(agg)

pm = pd.concat([pd.read_csv(p) for p in sorted(RESULTS_DIR.glob("fold_*_patient_metrics.csv"))])
pm.to_csv(RESULTS_DIR / "all_patient_metrics.csv", index=False)
print("Per classe (media su pazienti, NaN HD esclusi):")
display(pd.DataFrame({
    "Dice": [pm[f"dice_c{c}"].mean() for c in (1, 2, 3)], "IoU": [pm[f"iou_c{c}"].mean() for c in (1, 2, 3)],
    "HD (mm)": [pm[f"hd_c{c}"].mean() for c in (1, 2, 3)], "HD95 (mm)": [pm[f"hd95_c{c}"].mean() for c in (1, 2, 3)],
    "HD n/a": [pm[f"hd_c{c}"].isna().sum() for c in (1, 2, 3)]},
    index=["1 Necrotic core", "2 FLAIR hyperintensity", "3 Enhancing tumor"]))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for p in sorted(RESULTS_DIR.glob("fold_*_history.csv")):
    h, name = pd.read_csv(p), p.stem.replace("_history", "")
    axes[0, 0].plot(h["epoch"], h["train_loss"], label=name)
    axes[0, 1].plot(h["epoch"], h["val_loss"], label=name)
    axes[1, 0].plot(h["epoch"], h["val_dice"], label=name)
    axes[1, 1].plot(h["epoch"], h["val_iou"], label=name)
for ax, t in zip(axes.flat, ["Train loss", "Validation loss", "Validation Dice", "Validation IoU"]):
    ax.set_title(t); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "5fold_training_curves.png", dpi=200); plt.show()

In [ ]:
# ================================================================
# CELL 13 (OPZIONALE) - TEST SET INDIPENDENTE CON ENSEMBLE DEI FOLD
#   Imposta RUN_TEST_EVAL = True nella Cell 2. Il test set NON è mai usato per training né model selection.
# ================================================================
if RUN_TEST_EVAL:
    TEST_META = preprocess_dataframe(test_df, workers=min(4, os.cpu_count() or 1))
    META.update(TEST_META)

    models = []
    for p in sorted(CHECKPOINT_DIR.glob("nnunet_fold*_best.pth")):
        ck = torch.load(p, map_location=DEVICE, weights_only=False)
        m = NNUNet3D(NUM_INPUT_CHANNELS, NUM_CLASSES, PATCH_SIZE, BASE_FEATURES, MAX_FEATURES,
                     MIN_FEATURE_MAP, DEEP_SUPERVISION).to(DEVICE)
        m.load_state_dict(ck["model_state_dict"]); m.eval(); models.append(m)
    print(f"Ensemble di {len(models)} modelli")

    crit, rows = CombinedLoss(0.5, 0.5), []
    with torch.no_grad():
        for pid in tqdm([str(p) for p in test_df["patient_folder"]], desc="Test"):
            image, seg = load_patient_tensors(pid)
            prob = sum(torch.softmax(sliding_logits(m, image, overlap=0.5), dim=1) for m in models) / len(models)
            pred = prob.argmax(1)
            dices, ious = dice_iou_per_class(pred, seg)
            row = {"patient": pid, "dice": float(np.mean(dices)), "iou": float(np.mean(ious))}
            p_np, g_np = pred[0].cpu().numpy(), seg[0].cpu().numpy()
            for c in range(1, NUM_CLASSES):
                row[f"dice_c{c}"], row[f"iou_c{c}"] = dices[c - 1], ious[c - 1]
                row[f"hd_c{c}"], row[f"hd95_c{c}"] = hausdorff_metrics(p_np == c, g_np == c, META[pid]["spacing"])
            rows.append(row)
    test_res = pd.DataFrame(rows)
    test_res.to_csv(RESULTS_DIR / "test_patient_metrics.csv", index=False)
    display(test_res.drop(columns="patient").agg(["mean", "std"]).T)
else:
    print("RUN_TEST_EVAL = False: valutazione sul test set saltata.")